## Import Libraries

In [ ]:
# langgraph/langchain
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain.agents import Tool
from langchain_community.utilities import GoogleSerperAPIWrapper

# memory.
from langgraph.checkpoint.memory import MemorySaver
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# env
import os
import requests
from dotenv import load_dotenv
from typing import Annotated
from IPython.display import Image, display
import gradio as gr
from typing import TypedDict

## Initialize Variables

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

## Define State

In this example, we will be passing just messages from node to node; super step to super step

In [ ]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

## Define Tools

### Tool 1: Internet Searches

In [ ]:
# function.
serper = GoogleSerperAPIWrapper()

# tool.
tool_search = Tool(
    name="search",
    func=serper.run,
    description="Useful for when you need more information from an online search"
)

### Tool 2: Push Notifications

In [ ]:
# function.
def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

# tool.
tool_push = Tool(
    name="Push",
    func=push,
    description="Use this tool for sending a push notification (text) to the user"
)

In [ ]:
tools = [tool_search, tool_push]

## Define Memory

We will be using built-in memory and/or SQL

### Memory 1 - Default

In [ ]:
memory = MemorySaver()

### Memory 2 - SQL DB

In [ ]:
db_path = "memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)

## LLM

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Define Graph

In [ ]:
graph_builder = StateGraph(State)

# nodes.
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# edges.
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(checkpointer=memory)    # can use any of the memories I have defined
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable" : {"thread_id" : "1"}}

In [ ]:
def chat(user_input: str, history):
    result = graph.invoke({"messages" : [{"role" : "user", "content" : user_input}]}, config=config)
    return result["messages"][-1].content

In [ ]:
gr.ChatInterface(chat, type="messages").launch()